# How to define custom neural nets

`sbi` allows you to specify a specific density estimator for each of the implemented methods.
We support a variety of density estimators, e.g., mixtures of Gaussians, normalizing
flows, and diffusion models. Some of the density estimators are implemented as part of
`sbi`, for others we rely on other packages like
[`nflows`](https://github.com/bayesiains/nflows/) (via `pyknos`) or [`zuko`](https://github.com/probabilists/zuko).

For all options, check the API reference
[here](https://sbi.readthedocs.io/en/latest/sbi.html).


## Changing the type of density estimator

One option is using one of the preconfigured density estimators by passing a string in
the `density_estimator` keyword argument to the inference object (`NPE` or `NLE`), e.g.,
"maf" for a Masked Autoregressive Flow, of "nsf" for a Neural Spline Flow with default
hyperparameters.

**New with sbi 0.23:** Note that `"maf"` or `"nsf"` correspond to `nflows` density
estimators. Those have proven to work well, but the `nflows` package is not maintained
anymore. To use more recent and actively maintained density estimators, we tentatively
recommend using `zuko`, e.g., by passing `zuko_maf` or `zuko_nsf`.


In [1]:
!pip install sbi

In [2]:
import torch

from sbi.inference import NPE, NRE
from sbi.utils import BoxUniform

In [10]:
import torch
from sbi.inference import NPE, simulate_for_sbi
from sbi.utils import BoxUniform
from sbi.analysis import ActiveSubspace

# Define a simple simulator function for demonstration
def simulator(theta):
    # For a 2D theta, return a 2D observation
    # Adding some noise to simulate measurement/observation error
    return theta + torch.randn(theta.shape) * 0.1

prior = BoxUniform(torch.zeros(2), torch.ones(2))
inference = NPE(prior=prior, density_estimator="zuko_maf")

# Simulate some data and train the inference model
num_simulations = 100 # A small number for quick demonstration
theta_samples = prior.sample((num_simulations,))
x_observations = torch.stack([simulator(t) for t in theta_samples]) # Simulate observations

# Train the inference model
# The density_estimator is built and trained here, initializing self._neural_net
density_estimator_trained = inference.append_simulations(theta_samples, x_observations).train()

# Now build the posterior using the trained density estimator
posterior = inference.build_posterior(density_estimator_trained).set_default_x(torch.zeros(2))

# The rest of the original code
sensitivity = ActiveSubspace(posterior)
e_vals, e_vecs = sensitivity.find_directions(posterior_log_prob_as_property=True)


 Neural network successfully converged after 21 epochs.

  0%|          | 0/1000 [00:00<?, ?it/s]

In [22]:
import torch
from sbi.inference import NPE
from sbi.utils import BoxUniform
from sbi.analysis import ActiveSubspace
import pandas as pd
import numpy as np
import ast # Import the ast module for literal_eval

# Load theta and x from the created CSV files
# Assuming theta_data.csv is correctly formatted with numeric values throughout
loaded_theta_samples = torch.tensor(pd.read_csv('theta_data.csv').values.astype(float), dtype=torch.float32)

# For x_data.csv, directly parse string representations of lists
df_x = pd.read_csv('x_data.csv', header=None) # Read without a header



# Convert the list of lists/tuples to a single NumPy array
loaded_x_observations = torch.tensor(np.array(list_of_arrays_x), dtype=torch.float32)


# Define prior and inference object as before
prior = BoxUniform(torch.zeros(2), torch.ones(2))
inference = NPE(prior=prior, density_estimator="zuko_maf")

# Append the loaded simulations and train the inference model
density_estimator_trained = inference.append_simulations(
    loaded_theta_samples, loaded_x_observations
).train()

# Build the posterior
# Set default_x to match the observed data's dimension
posterior = inference.build_posterior(density_estimator_trained).set_default_x(torch.zeros(initial_length))

# You can then use the posterior for analysis
sensitivity = ActiveSubspace(posterior)
e_vals, e_vecs = sensitivity.find_directions(posterior_log_prob_as_property=True)

print("Model trained and posterior built using data loaded from files.")
print(f"Eigenvalues (sensitivity directions): {e_vals}")

 Neural network successfully converged after 63 epochs.

  0%|          | 0/1000 [00:00<?, ?it/s]

RuntimeError: The size of tensor a (1000) must match the size of tensor b (2) at non-singleton dimension 0

In [24]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In the case of `NRE`, the argument is called `classifier`:


In [5]:
inference = NRE(prior=prior, classifier="resnet")

## Changing hyperparameters of density estimators


Alternatively, you can use a set of utils functions to configure a density estimator yourself, e.g., use a MAF with hyperparameters chosen for your problem at hand.

Here, because we want to use N*P*E, we specifiy a neural network targeting the _posterior_ (using the utils function `posterior_nn`). In this example, we will create a neural spline flow (`'nsf'`) with `60` hidden units and `3` transform layers:


In [ ]:
# For SNLE: likelihood_nn(). For SNRE: classifier_nn()
from sbi.neural_nets import posterior_nn

density_estimator_build_fun = posterior_nn(
    model="zuko_nsf", hidden_features=60, num_transforms=3
)
inference = NPE(prior=prior, density_estimator=density_estimator_build_fun)

It is also possible to pass an `embedding_net` to `posterior_nn()` to automatically
learn summary statistics from high-dimensional simulation outputs. You can find a more
detailed tutorial on this in [04_embedding_networks](https://sbi.readthedocs.io/en/latest/how_to_guide/04_embedding_networks.html).


## Building new density estimators from scratch


Finally, it is also possible to implement your own density estimator from scratch, e.g., including embedding nets to preprocess data, or to a density estimator architecture of your choice.

For this, the `density_estimator` argument needs to be a function that takes `theta` and `x` batches as arguments to then construct the density estimator after the first set of simulations was generated. Our factory functions in `sbi/neural_nets/factory.py` return such a function.

The returned `density_estimator` object needs to be a subclass of `DensityEstimator`, which requires to implement three methods:
    
- `log_prob(input, condition, **kwargs)`: Return the log probabilities of the inputs given a condition or multiple i.e. batched conditions.
- `loss(input, condition, **kwargs)`: Return the loss for training the density estimator.
- `sample(sample_shape, condition, **kwargs)`: Return samples from the density estimator.

See more information on the [Reference API page](https://sbi.readthedocs.io/en/latest/sbi.html).